# Experiment: Momentum resolution quicklook

This notebook runs a small chunk (100 events) through the momentum resolution runner and produces a few diagnostic plots.

- Settings: `layouts/u2_debug_material.yaml`
- Kinematics: `samples/bsmumu_rdf.parquet`
- Output: `outputs/quicklook_0_100.root`

> Requires an environment with **ROOT/PyROOT** available.


In [ ]:
# Setup
import os
from pathlib import Path

REPO = Path.cwd().resolve()
os.environ["PYTHONPATH"] = str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")

print("Repo:", REPO)
print("PYTHONPATH:", os.environ["PYTHONPATH"])


## 1) Run the runner (100 events)

This executes the same code path as the CLI runner and writes a ROOT `TNtuple` called `ntuple`.


In [ ]:
import subprocess

cmd = [
    "python3",
    "src/Apps/run_sampled_tracks.py",
    "--kinematic_file", "samples/bsmumu_rdf.parquet",
    "--settings_file", "layouts/u2_debug_material.yaml",
    "--output_file", "outputs/quicklook_0_100.root",
    "--start_entry", "0",
    "--stop_entry", "100",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 2) Read output and plot quicklooks

We plot a resolution-like observable (`fwd_pres_LastMeasurement`) versus momentum.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ROOT

f = ROOT.TFile.Open("outputs/quicklook_0_100.root", "READ")
assert f and not f.IsZombie(), "Failed to open output ROOT file"
t = f.Get("ntuple")
assert t, "Cannot find TNtuple named 'ntuple'"

p = []
pres = []
for e in t:
    p.append(float(e.p))
    pres.append(float(getattr(e, "fwd_pres_LastMeasurement")))

p = np.asarray(p)
pres = np.asarray(pres)
mask = np.isfinite(p) & np.isfinite(pres) & (p > 0) & (pres > 0)
p = p[mask]
pres = pres[mask]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.hist2d(p/1000.0, pres, bins=(80, 80), cmap="viridis")
ax.set_xlabel("p [GeV]")
ax.set_ylabel("fwd_pres_LastMeasurement")
ax.set_title("Quicklook: resolution observable vs p")
fig.colorbar(ax.collections[0], ax=ax, label="counts")
plt.show()


## 3) Optional: save plots to `outputs/`


In [ ]:
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# Re-make the last figure and save it
import matplotlib.pyplot as plt
plt.gcf().savefig(out_dir / "quicklook_pres_vs_p.png", dpi=160)
print("Wrote", out_dir / "quicklook_pres_vs_p.png")
